# AR Aging Analysis — pandas cross-check

The [app](https://barbarzilay100-coder.github.io/ar-aging-analysis/) generates a synthetic
receivables ledger in **JavaScript** (seeded RNG, dates relative to *today*) and computes every
dashboard number with **SQL** in the browser. This notebook reconciles that against an
**independent Python implementation**:

1. Load a dated JSON snapshot exported from the JS generator (`data/ledger-snapshot.json`).
2. Regenerate the same ledger with `generator.py` — a from-scratch Python port of the seeded
   generator — and assert the two are **identical row-by-row**.
3. Reproduce every dashboard KPI, the customer × bucket aging pivot, and the payment
   reconciliation with **pandas**.

Two implementations, one book — the cross-check is itself a reconciliation exercise.

*Note: amounts use the app's fallback FX rates so both sides are deterministic; the live site
converts non-ILS invoices at live FX, which only scales those amounts.*

In [1]:
import json
from datetime import date, timedelta

import pandas as pd

snap = json.load(open("data/ledger-snapshot.json"))
AS_OF = date.fromisoformat(snap["as_of"])

customers = pd.DataFrame(snap["customers"])
deals = pd.DataFrame(snap["deals"])
payments = pd.DataFrame(snap["payments"])
for col in ("issue_date", "due_date", "financed_date", "repaid_date"):
    deals[col] = pd.to_datetime(deals[col])
payments["received_date"] = pd.to_datetime(payments["received_date"])

print(f"snapshot as of {AS_OF}: {len(customers)} customers, {len(deals)} invoices, {len(payments)} payments")

snapshot as of 2026-07-16: 30 customers, 180 invoices, 180 payments


## 1 · Replicate the ledger in Python

`generator.py` re-implements the JS generator from scratch — same mulberry32 seed, same draw
order, JS `Math.round` semantics. If any of the ~4,000 random draws were consumed in a
different order, the ledgers would diverge immediately.

In [2]:
from generator import generate

replica = generate(AS_OF)

for table in ("customers", "deals", "payments"):
    assert snap[table] == replica[table], f"{table}: replica diverges from the JS snapshot!"
    print(f"{table:<9} {len(snap[table]):>4} rows — Python replica identical to the JS export")

customers   30 rows — Python replica identical to the JS export
deals      180 rows — Python replica identical to the JS export
payments   180 rows — Python replica identical to the JS export


## 2 · Reproduce the dashboard KPIs

Same definitions as the SQL that drives the dashboard (see `sql/queries.sql`, section A).

In [3]:
OPEN = deals.status.isin(["Financed", "Overdue"])
BOOK = deals.status.isin(["Financed", "Overdue", "Repaid"])
as_of_ts = pd.Timestamp(AS_OF)

sales_90d = deals.loc[deals.issue_date >= as_of_ts - timedelta(days=90), "invoice_amount"].sum()
dso_90d = round(deals.loc[OPEN, "invoice_amount"].sum() * 90 / sales_90d)
days_open = (deals.loc[BOOK, "repaid_date"].fillna(as_of_ts) - deals.loc[BOOK, "issue_date"]).dt.days
repaid_n, overdue_n = (deals.status == "Repaid").sum(), (deals.status == "Overdue").sum()

kpis = pd.DataFrame([
    ("Financed volume (ILS)", f"{deals.loc[BOOK, 'advance_amount'].sum():,.0f}"),
    ("Open exposure (ILS)", f"{deals.loc[OPEN, 'advance_amount'].sum():,.0f}"),
    ("Overdue (ILS / count)", f"{deals.loc[deals.status == 'Overdue', 'advance_amount'].sum():,.0f} / {overdue_n}"),
    ("Avg advance rate", f"{deals.advance_rate.mean() * 100:.1f}%"),
    ("Avg days to finance", f"{(deals.financed_date - deals.issue_date).dt.days.mean():.1f}"),
    ("DSO (90-day)", f"{dso_90d} d"),
    ("Avg days to collect", f"{days_open.mean():.0f} d"),
    ("Repayment rate", f"{repaid_n / (repaid_n + overdue_n) * 100:.0f}%"),
], columns=["KPI", "pandas value"])
kpis

,KPI,pandas value
0,Financed volume (ILS),"34,625,654"
1,Open exposure (ILS),"7,670,150"
2,Overdue (ILS / count),"4,618,125 / 17"
3,Avg advance rate,86.9%
4,Avg days to finance,3.9
5,DSO (90-day),72 d
6,Avg days to collect,79 d
7,Repayment rate,89%


## 3 · The aging pivot, customer × bucket

The signature AR report (the app builds it as a `SUM(CASE …)` SQL pivot — queries.sql A11);
here it is as a one-line `pivot_table`.

In [4]:
open_deals = deals[OPEN].merge(customers[["customer_id", "customer_name"]], on="customer_id")
open_deals["days_overdue"] = (as_of_ts - open_deals.due_date).dt.days
open_deals["bucket"] = pd.cut(open_deals.days_overdue, [-10**6, 0, 30, 60, 90, 10**6],
                              labels=["Current", "1-30", "31-60", "61-90", "90+"])
aging = open_deals.pivot_table(index="customer_name", columns="bucket", values="advance_amount",
                               aggfunc="sum", fill_value=0, observed=False)
aging.columns = aging.columns.astype(str)
aging["Total"] = aging.sum(axis=1)
aging = aging.sort_values("Total", ascending=False).round(0).astype(int)

# the pivot must tie out to the open-exposure KPI
assert aging["Total"].sum() == round(deals.loc[OPEN, "advance_amount"].sum())
print(f"grand total {aging['Total'].sum():,} — ties to the open-exposure KPI")
aging.head(10)

grand total 7,670,150 — ties to the open-exposure KPI


bucket,Current,1-30,31-60,61-90,90+,Total
customer_name,,,,,,
Yizrael Import Ltd,88200,0,0,0,1411020,1499220
Emek Systems Ltd,1442670,0,0,0,0,1442670
Ramon Plastics Ltd,1189350,0,0,0,0,1189350
Tavor Distribution Ltd,0,0,0,0,807840,807840
Negev Foods Ltd,0,0,0,0,636930,636930
Arava Steel Ltd,29750,0,0,0,374205,403955
Ayalon Agro Ltd,0,0,0,0,398430,398430
Galil Plastics Ltd,84420,0,0,0,131670,216090
Ramon Logistics Ltd,0,0,0,0,208320,208320


## 4 · Payment reconciliation

Mirrors the app's Reconciliation tab (queries.sql, section C): every settled invoice classified
by how its cash receipts tie to the invoice amount, plus aged unapplied cash.

In [5]:
applied = payments.dropna(subset=["deal_id"]).groupby("deal_id")["amount"].agg(paid="sum", receipts="count")
recon = deals[deals.status.isin(["Repaid", "Overdue"])].set_index("deal_id").join(applied)
recon[["paid", "receipts"]] = recon[["paid", "receipts"]].fillna(0)
recon["variance"] = recon.paid - recon.invoice_amount
recon["match_status"] = (
    recon.assign(match_status="Matched").match_status
    .mask(recon.receipts == 0, "Unpaid")
    .mask((recon.receipts > 0) & (recon.variance < -1), "Short-paid")
    .mask((recon.receipts > 0) & (recon.variance > 1), "Overpaid"))

In [6]:
# every Repaid invoice must be fully paid (or deliberately over)
assert (recon.loc[recon.status == "Repaid", "paid"] >= recon.loc[recon.status == "Repaid", "invoice_amount"] - 1).all()

summary = recon.groupby("match_status", observed=True).agg(
    invoices=("invoice_amount", "count"), invoice_ils=("invoice_amount", "sum"), variance_ils=("variance", "sum")).round(0).astype(int)

unapplied = payments[payments.deal_id.isna()].copy()
unapplied["days_unapplied"] = (as_of_ts - unapplied.received_date).dt.days
print(f"unapplied cash: {unapplied.amount.sum():,.0f} ILS across {len(unapplied)} remittances")
display(summary)
unapplied[["reference", "payer", "amount", "received_date", "days_unapplied"]].sort_values("days_unapplied", ascending=False)

unapplied cash: 207,400 ILS across 4 remittances


,invoices,invoice_ils,variance_ils
match_status,,,
Matched,122,27227100,0
Overpaid,15,3798700,1551511
Short-paid,4,672100,-345550
Unpaid,13,4516900,-4516900


,reference,payer,amount,received_date,days_unapplied
179,INV-2026-9626,BlueHarbor Trading,120900,2026-04-08,99
177,INV-2026-9603,GreenField Agro,19200,2026-04-09,98
178,INV-2026-9104,UrbanBuild,21900,2026-04-23,84
176,INV-2026-9608,FreshLine Foods,45400,2026-06-07,39


## Conclusion

* The Python replica reproduces the JavaScript ledger **bit-for-bit** — 30 customers,
  180 invoices, 180 payments identical, which validates both implementations against the shared
  specification (seed, draw order, rounding semantics).
* Every dashboard KPI, the aging pivot and the payment-reconciliation summary are reproduced
  independently in pandas from the same snapshot.
* The snapshot is dated (**as-of** above): the app regenerates its ledger relative to *today*,
  so re-export `data/ledger-snapshot.json` (see `export_snapshot.js`) to refresh, then re-run
  this notebook.